# Why Spark? A hands-on Pandas vs Spark comparison

We keep doing everything with `pandas` and it always seems to "just work" - so why bother with Spark?

This notebook answers that with code, not theory. We will generate one dataset, then run the
same three tasks twice: once with plain Python/pandas, once with Spark, and compare what
actually happens.

> Run this in a Fabric notebook attached to a Lakehouse. It needs both the `spark` session
> and the Lakehouse's `Files` area.


In [ ]:
# --- Tunable knobs ---
# If your pool feels slow, lower these. If everything finishes "too fast" to notice
# a difference, raise them - that's the whole point of this notebook.
NUM_FILES = 10
ROWS_PER_FILE = 50_000        # total rows = NUM_FILES * ROWS_PER_FILE
LOOP_ITERS = 300              # simulated "heavy" per-row work used in Round 2

DEMO_PATH_LOCAL = "/lakehouse/default/Files/pandas_vs_spark_demo"   # how plain Python/pandas sees it
DEMO_PATH_SPARK = "Files/pandas_vs_spark_demo"                       # how Spark sees the same folder


In [ ]:
# Generate the demo data with plain Python + pandas and land it in the Lakehouse.
#
# Fabric mounts the attached Lakehouse locally at /lakehouse/default/, which is the only reason
# plain pandas (not just Spark) can reach OneLake files at all. Spark instead resolves the
# relative path "Files/..." against the attached Lakehouse directly. Same physical files, two
# different access routes - this is the "Storage Connection" idea from the architecture lesson.
import os
import numpy as np
import pandas as pd

os.makedirs(DEMO_PATH_LOCAL, exist_ok=True)

categories = ["Mountain Bikes", "Road Bikes", "Touring Bikes", "Accessories", "Helmets"]
rng = np.random.default_rng(seed=42)

total_rows = 0
for file_index in range(NUM_FILES):
    chunk = pd.DataFrame({
        "OrderID": range(file_index * ROWS_PER_FILE, (file_index + 1) * ROWS_PER_FILE),
        "Category": rng.choice(categories, ROWS_PER_FILE),
        "Quantity": rng.integers(1, 10, ROWS_PER_FILE),
        "UnitPrice": rng.uniform(20, 3000, ROWS_PER_FILE).round(2),
    })
    chunk.to_csv(f"{DEMO_PATH_LOCAL}/orders_{file_index}.csv", index=False)
    total_rows += len(chunk)

print(f"Wrote {NUM_FILES} files, {total_rows:,} rows total, to {DEMO_PATH_LOCAL}")


## Round 1: loading many files at once

Real ingestion almost never hands you one tidy file, it's a folder of them (one per day, one per
export, whatever). Let's load all 10 files, the pandas way and the Spark way.


In [ ]:
# The pandas way: YOU find the files, YOU loop over them, YOU glue them together.
import glob
import time

start = time.time()

files = glob.glob(f"{DEMO_PATH_LOCAL}/*.csv")
frames = [pd.read_csv(f) for f in files]
pdf = pd.concat(frames, ignore_index=True)

elapsed = time.time() - start
print(f"pandas: loaded {len(pdf):,} rows from {len(files)} files in {elapsed:.2f}s")
pdf.head()


In [ ]:
# The Spark way: one wildcard path. Spark finds the files AND reads them in parallel
# across the pool's cores - the file-discovery and merging code you had to write above
# is just... gone.
start = time.time()

sdf = (spark.read.format("csv")
       .option("header", True)
       .option("inferSchema", True)
       .load(f"{DEMO_PATH_SPARK}/*.csv"))

row_count = sdf.count()
elapsed = time.time() - start
print(f"Spark: loaded {row_count:,} rows in {elapsed:.2f}s")
display(sdf.limit(5))


Notice: same result, but the pandas version needed you to write the file-merging logic
yourself, running in a single Python process. The Spark version is one call, and behind the
scenes it was split across the worker nodes from the architecture lesson instead of run serially.


## Round 2: a computation that actually costs something

Loading data is cheap. Let's do real work per row, something CPU-bound, and see where a single
machine (pandas) starts to sweat compared to a pool of them (Spark).


In [ ]:
# pandas .apply(): Python evaluates this function ONE ROW AT A TIME, on ONE CPU core,
# no matter how many cores your machine has.
def compute_total_pandas(row):
    total = 0.0
    for _ in range(LOOP_ITERS):
        total += row["Quantity"] * row["UnitPrice"]
    return total / LOOP_ITERS

start = time.time()
pdf["ComputedTotal"] = pdf.apply(compute_total_pandas, axis=1)
elapsed = time.time() - start
print(f"pandas .apply() over {len(pdf):,} rows took {elapsed:.2f}s (single core)")


In [ ]:
# Spark UDF: the exact same row-by-row Python logic, but Spark hands out batches of rows
# to every executor/core in the pool and runs them at the same time.
from pyspark.sql.functions import udf, col
from pyspark.sql.types import DoubleType

def compute_total_spark(quantity, unit_price):
    total = 0.0
    for _ in range(LOOP_ITERS):
        total += quantity * unit_price
    return total / LOOP_ITERS

compute_total_udf = udf(compute_total_spark, DoubleType())

start = time.time()
sdf_udf = sdf.withColumn("ComputedTotal", compute_total_udf(col("Quantity"), col("UnitPrice")))
sdf_udf.count()   # force it to actually run - more on this in Round 3
elapsed = time.time() - start
print(f"Spark UDF over {row_count:,} rows took {elapsed:.2f}s (spread across all cores)")


In [ ]:
# Spark, done properly: drop the Python UDF entirely and use Spark's own built-in,
# vectorized column expressions. No per-row Python interpreter loop at all - this is
# the Catalyst Optimizer's native, columnar execution path.
start = time.time()
sdf_native = sdf.withColumn("ComputedTotal", col("Quantity") * col("UnitPrice"))
sdf_native.count()
elapsed = time.time() - start
print(f"Spark native column expression over {row_count:,} rows took {elapsed:.2f}s")


Notice the three timings. pandas did the work alone. The Spark UDF did the same work but
split across every core in the pool. The native Spark expression skipped Python row-by-row
execution entirely, which is why "avoid UDFs when a built-in function exists" is standard Spark
advice.


## Round 3: why nothing happens until you ask for it

pandas runs every line the instant it executes (eager). Spark stacks up transformations and
only actually runs them when you ask for a result (lazy), which is what lets the Catalyst
Optimizer plan the whole chain instead of executing it step by step.


In [ ]:
# Defining the transformation
start = time.time()
lazy_df = sdf.filter(col("Quantity") > 5).groupBy("Category").count()
print(f"Building the transformation took {time.time() - start:.4f}s - nothing has run yet")

# Triggering it with an action
start = time.time()
lazy_df.show()
print(f"Calling .show() took {time.time() - start:.2f}s - only now did Spark execute anything")


## Wrap-up

| | pandas | Spark |
| --- | --- | --- |
| Loading many files | manual glob + loop + concat | one wildcard path, read in parallel |
| Heavy per-row work | one CPU core | every core/executor in the pool |
| Best-case per-row work | still row-by-row Python | native columnar execution (no Python loop) |
| Execution model | eager, runs line by line | lazy, plans the whole chain then runs it |

So when do you actually reach for Spark? Not for a quick 50-row exploration, that's exactly
what a Starter Pool ad-hoc notebook is for. You reach for it when the data volume or the
per-row work is big enough that "one core, one machine" becomes the bottleneck, which is also
the point at which a Custom Spark Pool with autoscaling starts to matter.
